# FarmTech Solutions - Fase 5
### Previsão de Rendimento de Safra com Machine Learning
**Aluna:** Luana Brito da Silva  
**RM:** 566632

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans

print('Bibliotecas importadas com sucesso!')

## 2. Carregamento dos Dados

In [ ]:
df = pd.read_csv('crop_yield.csv')
df.columns = ['Cultura', 'Precipitacao', 'Umidade_Especifica', 'Umidade_Relativa', 'Temperatura', 'Rendimento']
print(f'Dataset com {df.shape[0]} linhas e {df.shape[1]} colunas')
df.head(10)

## 3. Análise Exploratória dos Dados
Exploramos os dados para entender o que temos: quantidade de registros, valores nulos, estatísticas e distribuição por cultura.

In [ ]:
print('Valores nulos:')
print(df.isnull().sum())
print('\nEstatísticas:')
print(df.describe().round(2))

In [ ]:
print('Culturas no dataset:')
print(df['Cultura'].value_counts())

In [ ]:
df.groupby('Cultura')['Rendimento'].mean().sort_values().plot(kind='barh', color='steelblue')
plt.title('Rendimento Médio por Cultura')
plt.xlabel('Rendimento Médio')
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(data=df, x='Cultura', y='Rendimento')
plt.title('Distribuição do Rendimento por Cultura')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
colunas = ['Precipitacao', 'Umidade_Especifica', 'Umidade_Relativa', 'Temperatura', 'Rendimento']
sns.heatmap(df[colunas].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlação entre Variáveis')
plt.tight_layout()
plt.show()

## 4. Clusterização com K-Means
O K-Means agrupa os dados por similaridade. Usamos o **Método do Cotovelo** para descobrir o número ideal de grupos e o **IQR** para detectar outliers.

In [ ]:
le = LabelEncoder()
df['Cultura_num'] = le.fit_transform(df['Cultura'])

X_c = df[['Precipitacao', 'Umidade_Especifica', 'Umidade_Relativa', 'Temperatura', 'Rendimento', 'Cultura_num']]
X_c_norm = StandardScaler().fit_transform(X_c)

inercias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_c_norm)
    inercias.append(km.inertia_)

plt.plot(range(2, 11), inercias, marker='o', color='steelblue')
plt.title('Método do Cotovelo')
plt.xlabel('Número de Clusters')
plt.ylabel('Inércia')
plt.show()

In [ ]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = km.fit_predict(X_c_norm)

print('Rendimento médio por cluster:')
print(df.groupby('Cluster')['Rendimento'].mean().round(2))

Q1 = df['Rendimento'].quantile(0.25)
Q3 = df['Rendimento'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['Rendimento'] < Q1 - 1.5*IQR) | (df['Rendimento'] > Q3 + 1.5*IQR)]
print(f'\nOutliers encontrados: {len(outliers)}')
print(outliers[['Cultura', 'Rendimento']])

## 5. Preparação dos Dados para os Modelos
Separamos os dados em treino (80%) e teste (20%) e normalizamos para os modelos funcionarem melhor.

In [ ]:
X = df[['Precipitacao', 'Umidade_Especifica', 'Umidade_Relativa', 'Temperatura', 'Cultura_num']]
y = df['Rendimento']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm  = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

## 6. Os 5 Modelos de Regressão
Treinamos e avaliamos 5 algoritmos. As métricas usadas são:
- **MAE**: erro médio absoluto
- **RMSE**: penaliza erros grandes
- **R²**: quanto mais próximo de 1, melhor

In [ ]:
def avaliar(nome, y_real, y_prev):
    mae  = mean_absolute_error(y_real, y_prev)
    rmse = np.sqrt(mean_squared_error(y_real, y_prev))
    r2   = r2_score(y_real, y_prev)
    print(f'\n{nome}')
    print(f'  MAE:  {mae:,.0f}')
    print(f'  RMSE: {rmse:,.0f}')
    print(f'  R2:   {r2:.4f}')
    return {'Modelo': nome, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R2': round(r2,4)}

resultados = []

# Modelo 1 - Regressão Linear
m1 = LinearRegression()
m1.fit(X_train_norm, y_train)
resultados.append(avaliar('Regressão Linear', y_test, m1.predict(X_test_norm)))

# Modelo 2 - Árvore de Decisão
m2 = DecisionTreeRegressor(max_depth=5, random_state=42)
m2.fit(X_train, y_train)
resultados.append(avaliar('Árvore de Decisão', y_test, m2.predict(X_test)))

# Modelo 3 - Random Forest
m3 = RandomForestRegressor(n_estimators=100, random_state=42)
m3.fit(X_train, y_train)
resultados.append(avaliar('Random Forest', y_test, m3.predict(X_test)))

# Modelo 4 - Gradient Boosting
m4 = GradientBoostingRegressor(n_estimators=100, random_state=42)
m4.fit(X_train, y_train)
resultados.append(avaliar('Gradient Boosting', y_test, m4.predict(X_test)))

# Modelo 5 - SVR
m5 = SVR(kernel='rbf', C=100)
m5.fit(X_train_norm, y_train)
resultados.append(avaliar('SVR', y_test, m5.predict(X_test_norm)))

## 7. Comparação Final dos Modelos

In [ ]:
tabela = pd.DataFrame(resultados).sort_values('R2', ascending=False).reset_index(drop=True)
print(tabela)

plt.barh(tabela['Modelo'], tabela['R2'], color='steelblue', edgecolor='black')
plt.title('Comparação de R2 entre os Modelos')
plt.xlabel('R2 (quanto maior, melhor)')
plt.tight_layout()
plt.show()

print(f"\nMelhor modelo: {tabela.iloc[0]['Modelo']} com R2 = {tabela.iloc[0]['R2']}")

## 8. Conclusões

- O dataset tem 155 registros com 4 culturas: Cocoa beans, Oil palm fruit, Rice paddy e Rubber natural.
- Não foram encontrados valores nulos — os dados estão limpos.
- O **Oil palm fruit** tem rendimento muito maior que as outras culturas.
- A clusterização com K-Means separou bem os grupos por cultura.
- Os modelos **Random Forest** e **Gradient Boosting** tiveram o melhor desempenho.

**Limitações:**
- Dataset pequeno (155 registros).
- Faltam variáveis de solo (pH, nutrientes) que também influenciam o rendimento.